# Sistema experto pedagógico: detección de plantas, diagnóstico de enfermedades y cuidados (con lógica)

Este notebook es una guía **paso a paso** para entender y aplicar:

- **Lógica proposicional**: operadores, inferencia, tablas de verdad.
- **Lógica de predicados**: cuantificadores, reglas, implicaciones.
- **Encadenamiento hacia adelante (forward chaining)** y **hacia atrás (backward chaining)**.
- **Representación de hechos, reglas y conocimiento** en sistemas expertos.
- **Ontologías básicas y grafos de conocimiento**.

## Problema a resolver (versión didáctica)
Queremos un *asistente* que:

1) **Identifique** una planta (a partir de rasgos simples, sin visión artificial por ahora).  
2) **Diagnostique** enfermedades probables (a partir de síntomas).  
3) Proponga **cuidados y acciones** (riego, luz, tratamiento, recomendaciones).

> Este notebook se enfoca en la **lógica e inferencia** (IA simbólica).  
> La parte de “detector” se implementa como reglas por rasgos observables.  
> Si luego quieres conectar visión artificial (CNN/YOLO), puedes usarla para extraer rasgos → alimentar este motor lógico.

---

## 0) Preparación

Usaremos bibliotecas estándar y, para grafo de conocimiento, **networkx** (y matplotlib para visualizar).

Si no tienes `networkx`, ejecuta la celda de instalación.

In [ ]:
# Si te falta networkx, descomenta:
# !pip -q install networkx

import itertools
from dataclasses import dataclass
from typing import Dict, List, Tuple, Set, Callable, Optional, Any

import networkx as nx
import matplotlib.pyplot as plt

---

# 1) Lógica proposicional

## 1.1 ¿Qué es una proposición?
Una **proposición** es una afirmación que puede ser **Verdadera (V)** o **Falsa (F)**.

Ejemplos en nuestro problema:

- **P**: “La hoja tiene manchas circulares”.
- **Q**: “Hay moho blanco en el envés”.
- **R**: “La planta está en exceso de humedad”.
- **D1**: “La enfermedad probable es oídio”.

## 1.2 Operadores lógicos (conectivos)
- Negación: **¬P** (NOT)
- Conjunción: **P ∧ Q** (AND)
- Disyunción: **P ∨ Q** (OR)
- Implicación: **P → Q** (“Si P entonces Q”)
- Bicondicional: **P ↔ Q** (“P si y solo si Q”)

En sistemas expertos, lo más usado es **implicación**:  
**(síntomas) → (conclusión)**.

In [ ]:
def NOT(p): 
    return not p

def AND(p, q): 
    return p and q

def OR(p, q): 
    return p or q

def IMPLIES(p, q):
    # P → Q es equivalente a (¬P) ∨ Q
    return (not p) or q

def IFF(p, q):
    # P ↔ Q
    return p == q

# Prueba rápida
P, Q = True, False
print("NOT(P) =", NOT(P))
print("P AND Q =", AND(P,Q))
print("P OR Q =", OR(P,Q))
print("P -> Q =", IMPLIES(P,Q))
print("P <-> Q =", IFF(P,Q))

## 1.3 Tablas de verdad

Una **tabla de verdad** evalúa una fórmula para todas las combinaciones posibles de V/F.

Ejemplo:

\[
(P \land Q) \rightarrow D1
\]

Interpretación:  
“Si P y Q son verdaderas, entonces concluyo D1”.

En la práctica:
- Si se cumplen **todos** los síntomas, disparo el diagnóstico (regla).
- Si NO se cumplen, la implicación se mantiene verdadera (porque la regla no se activa).

In [ ]:
def truth_table_2vars(formula: Callable[[bool,bool], bool], headers=("P","Q","Resultado")):
    rows = []
    for P,Q in itertools.product([False, True],[False, True]):
        rows.append((P,Q, formula(P,Q)))
    return headers, rows

# Ejemplo: (P∧Q)->D1 con D1=True (solo para ver comportamiento del operador)
headers, rows = truth_table_2vars(lambda P,Q: IMPLIES(AND(P,Q), True),
                                  headers=("P","Q","(P∧Q)->D1 (D1=True)"))
print(*headers, sep="\t")
for r in rows:
    print(*r, sep="\t")

### Nota (inferencia vs verificación)
- La **tabla de verdad** verifica una fórmula para valores asignados.
- La **inferencia** en un sistema experto usa reglas:  
  **si** (condiciones) **entonces** (conclusión).

Nosotros inferimos conclusiones **cuando las condiciones son verdaderas**.

---

# 2) Lógica de predicados (primer orden)

La lógica proposicional no “habla” de objetos (plantas específicas), solo de proposiciones globales.  
La lógica de predicados agrega:

- **Predicados**: propiedades/relaciones sobre objetos.
- **Variables**: x, y, …
- **Cuantificadores**:
  - ∀ (para todo)
  - ∃ (existe)

## 2.1 Ejemplos en el problema

- `Planta(x)` : x es una planta.
- `TieneSintoma(x, s)` : la planta x presenta el síntoma s.
- `Enfermedad(x, e)` : la planta x tiene la enfermedad e.
- `Requiere(x, c)` : x requiere cuidado c.

Ejemplos con cuantificadores:

- ∀x (Planta(x) → Requiere(x, Agua))  
  “Toda planta requiere agua (en alguna medida)”.

- ∃x (Planta(x) ∧ TieneSintoma(x, 'moho_blanco'))  
  “Existe una planta con moho blanco”.

In [ ]:
# Representaremos "objetos" como diccionarios para hacerlo didáctico.

plantas = [
    {"id": "p1", "nombre": "Planta_A", "tipo": None, "sintomas": {"moho_blanco", "hojas_deformadas"}},
    {"id": "p2", "nombre": "Planta_B", "tipo": None, "sintomas": {"manchas_circulares", "amarilleo"}},
]

def Planta(x): 
    return isinstance(x, dict) and "id" in x and "sintomas" in x

def TieneSintoma(x, s: str):
    return s in x.get("sintomas", set())

# ∀x: si es planta, entonces tiene atributo "sintomas" (por construcción)
forall_planta_tiene_sintomas = all((not Planta(x)) or ("sintomas" in x) for x in plantas)
print("∀x (Planta(x) -> tiene atributo 'sintomas') =", forall_planta_tiene_sintomas)

# ∃x: existe planta con moho_blanco
exists_moho_blanco = any(Planta(x) and TieneSintoma(x, "moho_blanco") for x in plantas)
print("∃x (Planta(x) ∧ TieneSintoma(x,'moho_blanco')) =", exists_moho_blanco)

## 2.2 Reglas como implicaciones con variables

Ejemplo de regla (predicados):

\[
\forall x: (TieneSintoma(x,\text{moho\_blanco}) \land TieneSintoma(x,\text{hojas\_deformadas})) \rightarrow Enfermedad(x,\text{oídio})
\]

Interpretación:  
Para cualquier planta x, si tiene esos síntomas, inferimos “oídio”.

En Python, lo implementaremos como reglas en un motor de inferencia.

---

# 3) Representación del conocimiento: hechos y reglas

En un sistema experto clásico:

- **Hechos**: información observada (inputs).
- **Reglas**: patrones condicionales (si-entonces).
- **Motor de inferencia**: aplica reglas a hechos para derivar nuevas conclusiones.

Construiremos un mini-motor con:
- una base de hechos por planta (síntomas, rasgos, contexto),
- reglas para **identificación** de planta,
- reglas para **diagnóstico**,
- reglas para **cuidados**.

In [ ]:
@dataclass(frozen=True)
class Fact:
    sujeto: str        # p.ej., "p1"
    predicado: str     # p.ej., "tiene_sintoma"
    objeto: str        # p.ej., "moho_blanco"

@dataclass
class Rule:
    nombre: str
    antecedentes: List[Tuple[str,str,str]]  # lista de (varSujeto, predicado, objeto) con varSujeto "$x" o id concreto
    consecuentes: List[Tuple[str,str,str]]  # idem
    condicion_extra: Optional[Callable[[Dict[str,str], Set[Fact]], bool]] = None

def fact(sujeto, predicado, objeto) -> Fact:
    return Fact(str(sujeto), str(predicado), str(objeto))

def match_pattern(pattern: Tuple[str,str,str], f: Fact, bindings: Dict[str,str]) -> Optional[Dict[str,str]]:
    ps, pp, po = pattern
    if pp != f.predicado or po != f.objeto:
        return None

    # sujeto variable
    if ps.startswith("$"):
        var = ps
        if var in bindings and bindings[var] != f.sujeto:
            return None
        nb = dict(bindings)
        nb[var] = f.sujeto
        return nb

    # sujeto constante
    if ps == f.sujeto:
        return dict(bindings)

    return None

def find_bindings_for_antecedents(antecedents: List[Tuple[str,str,str]], facts: Set[Fact]) -> List[Dict[str,str]]:
    results = []
    def backtrack(i: int, bindings: Dict[str,str]):
        if i == len(antecedents):
            results.append(bindings)
            return
        pat = antecedents[i]
        for f in facts:
            nb = match_pattern(pat, f, bindings)
            if nb is not None:
                backtrack(i+1, nb)
    backtrack(0, {})
    return results

def apply_rule(rule: Rule, facts: Set[Fact]) -> Set[Fact]:
    new_facts = set()
    bindings_list = find_bindings_for_antecedents(rule.antecedentes, facts)

    for b in bindings_list:
        if rule.condicion_extra and not rule.condicion_extra(b, facts):
            continue

        for (cs, cp, co) in rule.consecuentes:
            sujeto = b.get(cs, cs) if cs.startswith("$") else cs
            new_facts.add(fact(sujeto, cp, co))

    return new_facts - facts

def forward_chaining(rules: List[Rule], facts: Set[Fact], max_iter: int = 50, verbose: bool = True) -> Set[Fact]:
    all_facts = set(facts)
    for it in range(1, max_iter+1):
        added_any = False
        for r in rules:
            inferred = apply_rule(r, all_facts)
            if inferred:
                added_any = True
                all_facts |= inferred
                if verbose:
                    print(f"[Iter {it}] Regla '{r.nombre}' inferida:", inferred)
        if not added_any:
            if verbose:
                print(f"Detenido: no hay nuevos hechos (iteración {it}).")
            break
    return all_facts

---

# 4) Base de conocimiento del problema (plantas + síntomas + cuidados)

## 4.1 Síntomas y enfermedades (nivel didáctico)

Síntomas (ejemplos):
- `moho_blanco`
- `hojas_deformadas`
- `manchas_circulares`
- `amarilleo`
- `hojas_caidas`
- `puntos_negros`

Enfermedades:
- `oidio`
- `mancha_foliar`
- `pudricion_raiz`

Cuidados / acciones:
- `mejorar_ventilacion`
- `reducir_humedad`
- `fungicida_suave`
- `ajustar_riego`
- `evitar_encharcamiento`
- `retirar_hojas_afectadas`

## 4.2 Reglas del sistema experto

### A) Reglas de identificación (detector por rasgos)
Rasgos:
- `tiene_espinas`
- `hojas_gruesas`
- `hojas_ovales`
- `tallo_lenoso`
- `roseta`
- `tallo_suculento`

### B) Reglas de diagnóstico (síntomas → enfermedad)
- `moho_blanco ∧ hojas_deformadas` → `oidio`
- `manchas_circulares ∧ puntos_negros` → `mancha_foliar`
- `hojas_caidas ∧ amarilleo ∧ exceso_humedad` → `pudricion_raiz`

### C) Reglas de cuidado (enfermedad → acciones)
- `oidio` → mejorar ventilación, reducir humedad, fungicida suave
- `mancha_foliar` → retirar hojas afectadas, fungicida suave
- `pudricion_raiz` → ajustar riego, evitar encharcamiento

In [ ]:
# Reglas (A) Identificación
rules: List[Rule] = []

rules.append(Rule(
    nombre="Detecta_Cactus",
    antecedentes=[("$x","tiene_rasgo","tiene_espinas"),
                  ("$x","tiene_rasgo","tallo_suculento")],
    consecuentes=[("$x","es_planta","cactus")]
))

rules.append(Rule(
    nombre="Detecta_Suculenta",
    antecedentes=[("$x","tiene_rasgo","hojas_gruesas"),
                  ("$x","tiene_rasgo","roseta")],
    consecuentes=[("$x","es_planta","suculenta")]
))

rules.append(Rule(
    nombre="Detecta_Ficus",
    antecedentes=[("$x","tiene_rasgo","tallo_lenoso"),
                  ("$x","tiene_rasgo","hojas_ovales")],
    consecuentes=[("$x","es_planta","ficus")]
))

# Reglas (B) Diagnóstico
rules.append(Rule(
    nombre="Dx_Oidio",
    antecedentes=[("$x","tiene_sintoma","moho_blanco"),
                  ("$x","tiene_sintoma","hojas_deformadas")],
    consecuentes=[("$x","tiene_enfermedad","oidio")]
))

rules.append(Rule(
    nombre="Dx_Mancha_Foliar",
    antecedentes=[("$x","tiene_sintoma","manchas_circulares"),
                  ("$x","tiene_sintoma","puntos_negros")],
    consecuentes=[("$x","tiene_enfermedad","mancha_foliar")]
))

rules.append(Rule(
    nombre="Dx_Pudricion_Raiz",
    antecedentes=[("$x","tiene_sintoma","hojas_caidas"),
                  ("$x","tiene_sintoma","amarilleo"),
                  ("$x","tiene_contexto","exceso_humedad")],
    consecuentes=[("$x","tiene_enfermedad","pudricion_raiz")]
))

# Reglas (C) Cuidados
rules.append(Rule(
    nombre="Care_Oidio",
    antecedentes=[("$x","tiene_enfermedad","oidio")],
    consecuentes=[("$x","requiere_cuidado","mejorar_ventilacion"),
                  ("$x","requiere_cuidado","reducir_humedad"),
                  ("$x","requiere_cuidado","fungicida_suave")]
))

rules.append(Rule(
    nombre="Care_Mancha_Foliar",
    antecedentes=[("$x","tiene_enfermedad","mancha_foliar")],
    consecuentes=[("$x","requiere_cuidado","retirar_hojas_afectadas"),
                  ("$x","requiere_cuidado","fungicida_suave")]
))

rules.append(Rule(
    nombre="Care_Pudricion_Raiz",
    antecedentes=[("$x","tiene_enfermedad","pudricion_raiz")],
    consecuentes=[("$x","requiere_cuidado","ajustar_riego"),
                  ("$x","requiere_cuidado","evitar_encharcamiento")]
))

len(rules), [r.nombre for r in rules]

---

# 5) Caso de prueba: una planta observada (hechos iniciales)

**Ejemplo 1**  
- Rasgos: `hojas_gruesas`, `roseta`  
- Síntomas: `moho_blanco`, `hojas_deformadas`

Esperamos:
- Tipo: `suculenta`
- Diagnóstico: `oidio`
- Cuidados: mejorar ventilación, reducir humedad, fungicida suave

In [ ]:
facts0: Set[Fact] = set()

# Hechos de identificación
facts0 |= {
    fact("p1","tiene_rasgo","hojas_gruesas"),
    fact("p1","tiene_rasgo","roseta"),
}

# Hechos de síntomas
facts0 |= {
    fact("p1","tiene_sintoma","moho_blanco"),
    fact("p1","tiene_sintoma","hojas_deformadas"),
}

facts_all = forward_chaining(rules, facts0, verbose=True)

def facts_of(sujeto: str, predicado: str, facts: Set[Fact]) -> List[str]:
    return sorted({f.objeto for f in facts if f.sujeto==sujeto and f.predicado==predicado})

print("\n--- RESULTADOS ---")
print("Tipo de planta:", facts_of("p1","es_planta", facts_all))
print("Enfermedad:", facts_of("p1","tiene_enfermedad", facts_all))
print("Cuidados:", facts_of("p1","requiere_cuidado", facts_all))

## 5.1 ¿Qué pasó aquí?
Eso fue **encadenamiento hacia adelante**:
- partimos de hechos observados,
- aplicamos reglas,
- añadimos conclusiones como nuevos hechos,
- repetimos hasta que no haya nada nuevo.

---

# 6) Encadenamiento hacia atrás (backward chaining)

Proceso inverso:
- Tengo una **meta** (“¿p1 tiene oídio?”).
- Busco una regla que concluya esa meta.
- Intento probar sus antecedentes (sub-metas).

Esto sirve cuando el usuario pregunta algo específico y no quieres inferir “todo”.

In [ ]:
def backward_chaining(goal: Fact, rules: List[Rule], facts: Set[Fact], depth: int = 0, max_depth: int = 20) -> bool:
    if goal in facts:
        return True
    if depth >= max_depth:
        return False

    for r in rules:
        for cs, cp, co in r.consecuentes:
            if cp != goal.predicado or co != goal.objeto:
                continue

            bindings = {}
            if cs.startswith("$"):
                bindings[cs] = goal.sujeto
            elif cs != goal.sujeto:
                continue

            ok = True
            for (asuj, ap, ao) in r.antecedentes:
                suj_resuelto = bindings.get(asuj, asuj) if asuj.startswith("$") else asuj
                subgoal = fact(suj_resuelto, ap, ao)
                if not backward_chaining(subgoal, rules, facts, depth+1, max_depth):
                    ok = False
                    break
            if ok:
                return True
    return False

print("¿p1 tiene oidio?", backward_chaining(fact("p1","tiene_enfermedad","oidio"), rules, facts0))
print("¿p1 tiene pudricion_raiz?", backward_chaining(fact("p1","tiene_enfermedad","pudricion_raiz"), rules, facts0))

---

# 7) Cuantificadores (predicados) sobre múltiples plantas

Probamos inferencia con dos plantas: `p1` y `p2`, y verificamos una propiedad tipo ∀.

In [ ]:
facts_multi: Set[Fact] = set()

# p1
facts_multi |= {
    fact("p1","tiene_rasgo","hojas_gruesas"),
    fact("p1","tiene_rasgo","roseta"),
    fact("p1","tiene_sintoma","moho_blanco"),
    fact("p1","tiene_sintoma","hojas_deformadas"),
}

# p2 (mancha foliar)
facts_multi |= {
    fact("p2","tiene_rasgo","tallo_lenoso"),
    fact("p2","tiene_rasgo","hojas_ovales"),
    fact("p2","tiene_sintoma","manchas_circulares"),
    fact("p2","tiene_sintoma","puntos_negros"),
}

facts_multi_all = forward_chaining(rules, facts_multi, verbose=False)

def sujetos(facts: Set[Fact]) -> List[str]:
    return sorted({f.sujeto for f in facts})

def forall_oidio_implica_ventilacion(facts: Set[Fact]) -> bool:
    for x in sujetos(facts):
        has_oidio = fact(x,"tiene_enfermedad","oidio") in facts
        has_vent = fact(x,"requiere_cuidado","mejorar_ventilacion") in facts
        if has_oidio and not has_vent:
            return False
    return True

print("Sujetos:", sujetos(facts_multi_all))
print("∀x (oidio(x) -> mejorar_ventilacion(x)) =", forall_oidio_implica_ventilacion(facts_multi_all))

print("\nDiagnósticos:")
for x in sujetos(facts_multi_all):
    dx = facts_of(x, "tiene_enfermedad", facts_multi_all)
    if dx:
        print("-", x, "->", dx)

---

# 8) Ontologías básicas y grafos de conocimiento

Ontología mínima del dominio:
- Clases: Planta, Síntoma, Enfermedad, Cuidado
- Relaciones:
  - `tiene_sintoma(Planta, Síntoma)`
  - `tiene_enfermedad(Planta, Enfermedad)`
  - `requiere_cuidado(Planta, Cuidado)`
  - `sintoma_indica(Síntoma, Enfermedad)`
  - `enfermedad_requiere(Enfermedad, Cuidado)`

In [ ]:
G = nx.DiGraph()

def add_edge(s, rel, o):
    G.add_node(s)
    G.add_node(o)
    G.add_edge(s, o, rel=rel)

# Hechos inferidos -> aristas
for f in facts_multi_all:
    if f.predicado in {"tiene_sintoma","tiene_enfermedad","requiere_cuidado","es_planta"}:
        add_edge(f.sujeto, f.predicado, f.objeto)

# Conocimiento ontológico adicional (explicabilidad)
ontologia_relaciones = [
    ("moho_blanco", "sintoma_indica", "oidio"),
    ("hojas_deformadas", "sintoma_indica", "oidio"),
    ("manchas_circulares", "sintoma_indica", "mancha_foliar"),
    ("puntos_negros", "sintoma_indica", "mancha_foliar"),
    ("oidio", "enfermedad_requiere", "mejorar_ventilacion"),
    ("oidio", "enfermedad_requiere", "reducir_humedad"),
    ("oidio", "enfermedad_requiere", "fungicida_suave"),
    ("mancha_foliar", "enfermedad_requiere", "retirar_hojas_afectadas"),
    ("mancha_foliar", "enfermedad_requiere", "fungicida_suave"),
]
for s, rel, o in ontologia_relaciones:
    add_edge(s, rel, o)

print("Nodos:", len(G.nodes), "| Aristas:", len(G.edges))

In [ ]:
plt.figure(figsize=(12, 8))
pos = nx.spring_layout(G, seed=42)

nx.draw_networkx_nodes(G, pos, node_size=900, alpha=0.9)
nx.draw_networkx_labels(G, pos, font_size=9)

nx.draw_networkx_edges(G, pos, arrows=True, alpha=0.5)
edge_labels = {(u,v): G.edges[u,v]["rel"] for u,v in G.edges()}
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=7)

plt.axis("off")
plt.title("Grafo de conocimiento: plantas, síntomas, enfermedades y cuidados")
plt.show()

---

# 9) “Detector” de plantas (versión lógica, sin visión artificial)

Si una app detecta “espinas” y “tallo suculento”, el motor lógico concluye: `cactus`.

Probemos un ejemplo `p3`.

In [ ]:
facts_p3: Set[Fact] = {
    fact("p3","tiene_rasgo","tiene_espinas"),
    fact("p3","tiene_rasgo","tallo_suculento"),
}
facts_p3_all = forward_chaining(rules, facts_p3, verbose=False)
print("Tipo de planta p3:", facts_of("p3","es_planta", facts_p3_all))

---

# 10) Función utilitaria: diagnóstico + cuidados

Para hacerlo “usable”:
- recibo `rasgos`, `sintomas`, `contexto`,
- genero hechos,
- corro inferencia,
- regreso un reporte.

In [ ]:
def diagnosticar_planta(
    plant_id: str,
    rasgos: List[str],
    sintomas: List[str],
    contexto: Optional[List[str]] = None,
    verbose: bool = False
) -> Dict[str, Any]:
    contexto = contexto or []
    f: Set[Fact] = set()

    for r in rasgos:
        f.add(fact(plant_id, "tiene_rasgo", r))
    for s in sintomas:
        f.add(fact(plant_id, "tiene_sintoma", s))
    for c in contexto:
        f.add(fact(plant_id, "tiene_contexto", c))

    allf = forward_chaining(rules, f, verbose=verbose)

    return {
        "id": plant_id,
        "tipo_planta": facts_of(plant_id, "es_planta", allf),
        "enfermedades_probables": facts_of(plant_id, "tiene_enfermedad", allf),
        "cuidados_sugeridos": facts_of(plant_id, "requiere_cuidado", allf),
        "hechos_total": sorted(allf, key=lambda x: (x.sujeto, x.predicado, x.objeto)),
    }

reporte = diagnosticar_planta(
    "demo",
    rasgos=["hojas_gruesas","roseta"],
    sintomas=["moho_blanco","hojas_deformadas"],
    contexto=["exceso_humedad"],
    verbose=False
)

print("Tipo:", reporte["tipo_planta"])
print("Enfermedades:", reporte["enfermedades_probables"])
print("Cuidados:", reporte["cuidados_sugeridos"])

---

# 11) Ejercicios sugeridos

1) **Tablas de verdad**  
   Construye la tabla de verdad para:
   \[
   (P \lor Q) \land (\neg R) \rightarrow D
   \]
   e interpreta qué significa en contexto de síntomas.

2) **Nuevas reglas**  
   Agrega enfermedad `deficiencia_nutriente`:
   - `amarilleo ∧ crecimiento_lento` → `deficiencia_nutriente`  
   y cuidado:
   - `deficiencia_nutriente` → `abonar_equilibrado`

3) **Backward chaining**  
   Implementa la consulta:
   - “¿demo requiere fungicida_suave?”

4) **Grafo de conocimiento**  
   Agrega relación:
   - `tipo_planta_requiere_luz(tipo, nivel_luz)`  
   y consulta:
   - “¿Qué luz requiere una suculenta?”

---

# 12) Siguiente paso (si quieres escalar el proyecto)
Ideas de integración:

- **Visión artificial** (YOLO/CNN/CLIP) → extrae rasgos (espinas, roseta, etc.)
- Motor lógico (este notebook) → diagnóstico + recomendaciones + explicación
- Interfaz:
  - API con **FastAPI**
  - App con **Streamlit**
  - Agente con voz (STT/TTS)

Si me dices qué plantas te interesan (nombres comunes) y qué enfermedades quieres cubrir, ampliamos la base de conocimiento.